# Merchant Ranking
This notebook builds and evaluates merchant rankings using historical take rate, transaction volume, and dollar value. The ranking is constructed from the training period and evaluated on later data to avoid time leakage.


In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os, sys
import re
import glob
import math
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import pyspark.sql.functions as F
import sklearn
import traceback
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from datetime import datetime, timedelta
from collections import defaultdict
from multiprocessing import Manager
from xgboost import plot_importance, XGBRegressor
from pyspark.sql.window import Window
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV, train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, root_mean_squared_error

In [ ]:
sys.path.insert(0, "../scripts")
from spark_setup import get_spark
spark = get_spark()

# Load Dataset

In [ ]:
df = spark.read.parquet('.././data/curated/final_external_enriched')
df.show()

In [ ]:
df.printSchema()

# Merchant Summary
We reuse this for training later in the notebook

In [ ]:
def summarize_merchants(spark_df):
    return spark_df.groupBy("merchant_abn").agg(
        F.first("merchant_name", ignorenulls=True).alias("merchant_name"),
        F.first("take_rate", ignorenulls=True).alias("take_rate"),
        F.sum("dollar_value").alias("total_dollar_value"),
        F.count("order_id").alias("num_transactions"),
        F.avg("dollar_value").alias("avg_order_value"),
        F.avg("fraud_probability").alias("mean_fraud_probability"),
    )

# Full-period summary is retained only as a descriptive reference.
merchant_summary = summarize_merchants(df)

print(f"Merchants summarized: {merchant_summary.count():,}")
merchant_summary.select(
    "merchant_abn", "merchant_name", "take_rate",
    "total_dollar_value", "num_transactions",
    "avg_order_value", "mean_fraud_probability"
).show(truncate=False)


# Ranking

We rank based on take rate, dollar value and transaction volume. This reduces the chance that a merchant looks strong only because of one metric.

In [ ]:
def add_percentile_scores(summary_df):
    return (
        summary_df
        .withColumn(
            "take_rate_score",
            F.percent_rank().over(Window.orderBy(F.col("take_rate")))
        )
        .withColumn(
            "dollar_value_score",
            F.percent_rank().over(Window.orderBy(F.col("total_dollar_value")))
        )
        .withColumn(
            "transaction_score",
            F.percent_rank().over(Window.orderBy(F.col("num_transactions")))
        )
        .withColumn(
            "avg_order_value_score",
            F.percent_rank().over(Window.orderBy(F.col("avg_order_value")))
        )
    )

scored_summary = add_percentile_scores(merchant_summary)

In [ ]:
ranking = (
    scored_summary
    .withColumn(
        "ranking_score",
        (
            F.col("take_rate_score")
            + F.col("dollar_value_score")
            + F.col("transaction_score")
        ) / 3
    )
    .withColumn(
        "rank",
        F.row_number().over(Window.orderBy(F.desc("ranking_score")))
    )
    .orderBy(F.desc("ranking_score"))
)

print("Stronger ranking: take rate + dollar value + transaction volume")
ranking.select(
    "rank", "merchant_abn", "merchant_name", "take_rate",
    "total_dollar_value", "num_transactions", "avg_order_value",
    "ranking_score"
).show(20, truncate=False)


In [ ]:
# save rankings
ranking.write.mode("overwrite").parquet(
    "../data/curated/rankings_by_volume_score.parquet"
)


In [ ]:
plot = ranking.select(
    "take_rate_score", "transaction_score", "ranking_score"
).toPandas()

fig, ax = plt.subplots(figsize=(8, 6))

sc = ax.scatter(
    plot["take_rate_score"],
    plot["transaction_score"],
    c=plot["ranking_score"],
    cmap="magma_r",
    s=20,
    alpha=0.7,
)

ax.set_title("Take Rate vs Transaction Volume")
ax.set_xlabel("Take-rate percentile")
ax.set_ylabel("Transaction-volume percentile")

fig.colorbar(sc, ax=ax, label="Combined score")

plt.tight_layout()
plt.show()

# Train-Validate-Test Split

We learn ranking based on historical data then evaluate later data (no shuffle of time-series data)

In [ ]:
train_size = 0.70
val_size = 0.15
test_size = 1 - train_size - val_size

date_bounds = df.select(
    F.min("order_datetime").alias("min_date"),
    F.max("order_datetime").alias("max_date")
).collect()[0]

min_date = date_bounds["min_date"]
max_date = date_bounds["max_date"]
total_days = (max_date - min_date).days

train_end = min_date + timedelta(days=int(total_days * train_size))
val_end = min_date + timedelta(days=int(total_days * (train_size + val_size)))

train_df = df.filter(F.col("order_datetime") <= train_end)
val_df = df.filter(
    (F.col("order_datetime") > train_end) &
    (F.col("order_datetime") <= val_end)
)
test_df = df.filter(F.col("order_datetime") > val_end)

print(f"Date range: {min_date} to {max_date} ({total_days} days)")
print(f"Train:      {min_date} to {train_end}")
print(f"Validation: {train_end} to {val_end}")
print(f"Test:       {val_end} to {max_date}")
print()
print(f"Train rows:      {train_df.count():,}")
print(f"Validation rows: {val_df.count():,}")
print(f"Test rows:       {test_df.count():,}")


Build the ranking using train data only

In [ ]:
train_summary = summarize_merchants(train_df)
train_scored = add_percentile_scores(train_summary)

train_ranked_volume = (
    train_scored
    .withColumn(
        "ranking_score",
        (
            F.col("take_rate_score")
            + F.col("dollar_value_score")
            + F.col("transaction_score")
        ) / 3
    )
    .withColumn(
        "rank",
        F.row_number().over(Window.orderBy(F.desc("ranking_score")))
    )
)

print("Top 20 merchants using TRAIN-period information only:")
train_ranked_volume.orderBy("rank").select(
    "rank", "merchant_abn", "merchant_name", "take_rate",
    "total_dollar_value", "num_transactions", "avg_order_value",
    "mean_fraud_probability", "ranking_score"
).show(20, truncate=False)


# Evaluate ranking stability on the test period

We compare the train ranking against test-period transaction-volume ranks, using only merchants observed in both periods.

In [ ]:
def make_test_summary(spark_df):
    return spark_df.groupBy("merchant_abn").agg(
        F.sum("dollar_value").alias("test_total_dollar_value"),
        F.count("order_id").alias("test_num_transactions"),
    )


test_summary = make_test_summary(test_df)

# Add actual test-period ranks.
test_ranked = (
    test_summary
    .withColumn(
        "test_dollar_rank",
        F.row_number().over(Window.orderBy(F.desc("test_total_dollar_value")))
    )
    .withColumn(
        "test_transaction_rank",
        F.row_number().over(Window.orderBy(F.desc("test_num_transactions")))
    )
)

# Compare the two candidate TRAIN rankings with the same TEST merchants.
eval_df = (
    train_ranked_volume.select(
        "merchant_abn", "rank", "ranking_score"
    )
    .withColumnRenamed("rank", "train_rank")
    .withColumnRenamed("ranking_score", "train_score")
    .join(test_ranked, on="merchant_abn", how="inner")
)

eval_pd = eval_df.select(
    "merchant_abn",
    "train_rank",
    "train_score",
    "test_dollar_rank",
    "test_transaction_rank",
).toPandas()

# Spearman rank correlation: higher magnitude means stronger rank agreement.
dollar_corr = eval_pd["train_rank"].corr(
    eval_pd["test_dollar_rank"], method="spearman"
)
tx_corr = eval_pd["train_rank"].corr(
    eval_pd["test_transaction_rank"], method="spearman"
)

print("\nRank stability")
print(f"take rate + dollar value + transactions vs test dollar rank: "
      f"{dollar_corr:.4f}")
print(f"ranking vs test transaction rank: "
      f"{tx_corr:.4f}")

In [ ]:
# Top-K overlap measures whether the ranking identifies merchants that remain among the strongest performers later.
def top_k_overlap(train_rank_df, test_rank_df, k):
    train_top = {
        r["merchant_abn"]
        for r in train_rank_df.orderBy("rank").limit(k).select("merchant_abn").collect()
    }
    test_top = {
        r["merchant_abn"]
        for r in test_rank_df.orderBy("test_dollar_rank").limit(k).select("merchant_abn").collect()
    }
    return len(train_top.intersection(test_top)), len(train_top), len(test_top)

print("\nTop-K overlap: stronger TRAIN ranking vs TEST dollar-value ranking")
for k in [10, 25, 50, 100]:
    overlap, train_n, test_n = top_k_overlap(train_ranked_volume, test_ranked, k)
    print(f"Top {k:>3}: {overlap}/{k} merchants overlap")

# Visual check of rank stability.
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=eval_pd,
    x="train_rank",
    y="test_dollar_rank",
    alpha=0.6,
)
plt.title("Train Ranking vs Later Test-Period Ranking")
plt.xlabel("Rank from training period (1 = highest)")
plt.ylabel("Rank by test-period dollar value (1 = highest)")
plt.grid()
plt.show()
